In [2]:
import sys
sys.path.insert(0, "/myhome/smartt")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from matplotlib.lines import Line2D
import torch

from smartt.data_containers import get_dataset, REGISTRY
from smartt.saxs_naf.cache import (
    load_recon, list_cache, _param_hash, cache_table, CacheChooser, project_params,
)
from smartt.saxs_naf.eval import evaluate_real_sh
from smartt.saxs_fbp import fibonacci_hemisphere

## Configuration

Set `DATASET`, `DC_TYPE`, and `JOBS` to select which cached reconstructions to load.

`JOBS` is a list of `(label, method, params_override)` triples.  
The override dict merges with `_DEFAULTS`; any key from `_DEFAULTS` can be overridden.

In [4]:
DATASET    = "b411"   # one of: b411, zenodo, frogbone, fiber-synthetic
DC_TYPE    = "main"   # one of: main, remount, combined
ELL_MAX    = 8
K          = 30       # number of reciprocal-space directions for Viewer B
HALF_SPACE = 'y'      # hemisphere convention passed to fibonacci_hemisphere

_DEFAULTS = dict(
    ell_max         = ELL_MAX,
    # NAF
    n_iterations    = 3000,
    lr              = 0.002,
    batch_size      = 100,
    reg_weight_sh   = 1e-6,
    reg_weight_tv   = 5e-6,
    # mumott
    mumott_iters    = 20,
    laplacian_weight= 0.1,
    maxcor          = 5,
    # holdout
    holdout_frac    = 0.0,
    holdout_seed    = 42,
)

# (display label, method string, params override)
JOBS = [
    ("mumott_sh", "mumott_sh", {}),
    ("mumott_gk", "mumott_gk", {}),
    ("NAF",       "naf",       {}),
    ("FBP",       "fbp",       {}),
]

## Browse cache

In [ ]:
ds = get_dataset(DATASET)
cache_dir = ds.get_cache_dir()

# Browse everything already computed, grouped by method. Pick a dc_type, then a
# row per method (leave "none" to keep the Configuration-cell value). Click Load,
# then re-run the adapter cell below plus everything downstream.
chooser = CacheChooser(cache_dir)
chooser

In [6]:
# Apply the chooser selection to DC_TYPE and _DEFAULTS. Safe to run even if you
# never opened the chooser (falls back to the Configuration cell). _build_params
# below still pops the keys irrelevant to each method.
if 'chooser' in globals() and getattr(chooser, 'selection', None):
    sel = chooser.selection
    DC_TYPE = chooser.dc_type
    for _cat in ('naf', 'mumott_sh', 'mumott_gk'):
        if _cat in sel:
            _DEFAULTS.update(project_params(sel[_cat], _DEFAULTS))
    print('DC_TYPE  :', DC_TYPE)
    print('_DEFAULTS:', _DEFAULTS)
else:
    print('No chooser selection — using Configuration cell values.')

DC_TYPE  : main
_DEFAULTS: {'ell_max': 8, 'n_iterations': 2000, 'lr': 0.01, 'batch_size': 200, 'reg_weight_sh': 1e-06, 'reg_weight_tv': 0, 'mumott_iters': 20, 'laplacian_weight': 0.1, 'maxcor': 5, 'holdout_frac': 0.0, 'holdout_seed': 42}


## Load reconstructions

In [7]:
def _build_params(method, overrides):
    p = {**_DEFAULTS, **overrides}
    p["method"]  = method
    p["dataset"] = DATASET
    p["dc_type"] = DC_TYPE
    if method in ("mumott_sh", "mumott_gk"):
        for k in ("n_iterations", "lr", "batch_size", "reg_weight_sh", "reg_weight_tv"):
            p.pop(k, None)
    else:
        for k in ("mumott_iters", "laplacian_weight", "maxcor"):
            p.pop(k, None)
    return p


volumes = {}       # label -> (X, Y, Z, C_SH) float32
params_loaded = {}

for label, method, overrides in JOBS:
    params = _build_params(method, overrides)
    name   = f"{method}_{DATASET}_{DC_TYPE}"
    coeffs = load_recon(cache_dir, name, params)
    if coeffs is None:
        print(f"[missing]  {label:<15}  (hash: {_param_hash(params)})")
    else:
        volumes[label]       = coeffs
        params_loaded[label] = params
        print(f"[loaded]   {label:<15}  {coeffs.shape}  {coeffs.dtype}")

[loaded]   mumott_sh        (141, 111, 141, 45)  float32
[loaded]   mumott_gk        (141, 111, 141, 45)  float32
[loaded]   NAF              (141, 111, 141, 45)  float32
[missing]  FBP              (hash: b2df1d65)


## y-directions for Viewer B\n\nFibonacci-sampled hemisphere in reciprocal space — same convention as the FBP notebook.  \nAdjust `K` and `HALF_SPACE` in the configuration cell above.

In [8]:
y_directions = fibonacci_hemisphere(K, half_space=HALF_SPACE)  # (K, 3)
print(f"y_directions: {y_directions.shape}")
print(f"First 3:\n{np.round(y_directions[:3], 4)}")

y_directions: (30, 3)
First 3:
[[ 0.9999  0.0167 -0.    ]
 [-0.7364  0.05    0.6746]
 [ 0.0871  0.0833 -0.9927]]


---
## Viewer A — SH coefficients

Shows orthogonal slices of a selected SH coefficient index directly (no projection).  
Useful for inspecting individual basis components and their spatial structure.

In [ ]:
_crosshair_handles = [Line2D([], [], color=c, lw=1.5)
                      for c in ['limegreen', 'tomato', 'deepskyblue']]
_crosshair_labels  = ['x', 'y', 'z']


def make_volume_viewer(volumes):
    """Interactive 3-plane viewer for a dict of (X, Y, Z, C_SH) volumes."""
    if not volumes:
        print("No volumes loaded.")
        return

    keys    = list(volumes.keys())
    ref_key = keys[0]
    X_, Y_, Z_, C_ = volumes[ref_key].shape

    def _view(dataset, coeff, show_difference, x, y, z):
        vol   = volumes[dataset][..., coeff]
        ref_c = volumes[ref_key][..., coeff]

        lo   = float(np.percentile(ref_c, 1))
        hi   = float(np.percentile(ref_c, 99))
        dlim = max(abs(lo), abs(hi)) * 0.15

        if show_difference and dataset != ref_key:
            data     = vol - ref_c
            cmap     = 'RdBu_r'
            lo_, hi_ = -dlim, dlim
            clabel   = f'{dataset} − {ref_key}  [coeff {coeff}]'
        else:
            data     = vol
            cmap     = 'inferno'
            lo_, hi_ = lo, hi
            clabel   = f'coeff[{coeff}]'

        fig = plt.figure(figsize=(13, 5))
        gs  = fig.add_gridspec(1, 3, wspace=0.35)
        ax_yz = fig.add_subplot(gs[0, 0])
        ax_xz = fig.add_subplot(gs[0, 1])
        ax_xy = fig.add_subplot(gs[0, 2])

        kw = dict(cmap=cmap, vmin=lo_, vmax=hi_, aspect='equal', origin='lower')

        im = ax_yz.imshow(data[x, :, :].T, **kw)
        ax_yz.axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        ax_yz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_yz.set_title(f'YZ  (x={x})')
        ax_yz.set_xlabel('Y');  ax_yz.set_ylabel('Z')

        ax_xz.imshow(data[:, y, :].T, **kw)
        ax_xz.axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        ax_xz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_xz.set_title(f'XZ  (y={y})')
        ax_xz.set_xlabel('X');  ax_xz.set_ylabel('Z')

        ax_xy.imshow(data[:, :, z].T, **kw)
        ax_xy.axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        ax_xy.axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        ax_xy.set_title(f'XY  (z={z})')
        ax_xy.set_xlabel('X');  ax_xy.set_ylabel('Y')

        ax_xy.legend(_crosshair_handles, _crosshair_labels,
                     fontsize=7, loc='lower right', framealpha=0.5, title='slice pos.')
        plt.colorbar(im, ax=[ax_yz, ax_xz, ax_xy], shrink=0.65, label=clabel)
        plt.suptitle(f'{dataset}  |  coeff[{coeff}]  —  {DATASET}/{DC_TYPE}', fontsize=11)
        plt.show()

    interact(
        _view,
        dataset=widgets.Dropdown(
            options=keys, value=keys[0], description='Volume:',
            style={'description_width': 'initial'},
        ),
        coeff=widgets.IntSlider(
            min=0, max=C_-1, step=1, value=0,
            description='coeff idx', continuous_update=False,
        ),
        show_difference=widgets.Checkbox(value=False, description=f'Diff from {ref_key}'),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


make_volume_viewer(volumes)

interactive(children=(Dropdown(description='Volume:', options=('mumott_sh', 'mumott_gk', 'NAF'), style=Descrip…

---
## Viewer B — Directional intensities

Projects `(X, Y, Z, C_SH)` → `(K, X, Y, Z)` by evaluating the SH basis at each
`y_direction`, then shows the same 3-plane + histogram layout as the FBP comparison notebook.

The projection is: `I[k, x, y, z] = Σ_c  B[k, c] · coeff[x, y, z, c]`

Pass an optional `sphere_mask` (boolean array of shape `(X, Y, Z)`) to restrict the
histogram to interior voxels only.

In [ ]:
def make_directional_viewer(volumes, y_directions, ell_max=ELL_MAX, sphere_mask=None, show_histogram=False):
    """Project SH coefficients to directional intensities and show an interactive viewer."""
    if not volumes:
        print("No volumes loaded.")
        return

    keys    = list(volumes.keys())
    ref_key = keys[0]
    X_, Y_, Z_, C_ = volumes[ref_key].shape
    K_ = len(y_directions)

    # SH basis matrix  (K, C_SH)
    B = evaluate_real_sh(
        torch.tensor(y_directions, dtype=torch.float32), ell_max
    ).numpy()

    # Project all volumes once up front
    print("Projecting SH → directional intensities...")
    rt_volumes = {}
    for label, coeffs in volumes.items():
        flat = coeffs.reshape(-1, C_)          # (X*Y*Z, C)
        rt   = (flat @ B.T).reshape(X_, Y_, Z_, K_).transpose(3, 0, 1, 2)  # (K, X, Y, Z)
        rt_volumes[label] = rt.astype(np.float32)
        print(f"  {label}: {rt.shape}")

    ref_volume = rt_volumes[ref_key] if "GT" not in rt_volumes else rt_volumes['GT']


    # Color scale anchored to ref volume interior
    _abs_lo   = float(np.percentile(ref_volume, 0.5))
    _abs_hi   = float(np.percentile(ref_volume, 99.5))
    _diff_lim = 0.15 * (_abs_hi - _abs_lo)

    # Histogram bins from ref volume (all K)
    def _interior(vol_k):
        return vol_k[sphere_mask].ravel() if sphere_mask is not None else vol_k.ravel()

    ref_interiors = [_interior(ref_volume[k_]) for k_ in range(K_)]
    all_vals = np.concatenate(ref_interiors)
    bins = np.linspace(np.percentile(all_vals, 0.5), np.percentile(all_vals, 99.5), 80)

    def _view(dataset, show_difference, show_hist, k, x, y, z):
        vol   = rt_volumes[dataset]
        vol_k = vol[k]

        if show_difference and dataset != ref_key:
            data     = vol_k - ref_volume[k]
            cmap     = 'RdBu_r'
            lo, hi   = -_diff_lim, _diff_lim
            clabel   = f'{dataset} − {ref_key}'
        else:
            data     = vol_k
            cmap     = 'inferno'
            lo, hi   = _abs_lo, _abs_hi
            clabel   = 'Intensity'

        lo = min(float(np.percentile(ref_volume[k, x, :, :], 0.5)), float(np.percentile(ref_volume[k, :, y, :], 0.5)), float(np.percentile(ref_volume[k, :, :, z], 0.5)))
        hi = max(float(np.percentile(ref_volume[k, x, :, :], 99.9)), float(np.percentile(ref_volume[k, :, y, :], 99.9)), float(np.percentile(ref_volume[k, :, :, z], 99.9)))

        if show_hist:
            fig = plt.figure(figsize=(13, 9))
            gs  = fig.add_gridspec(2, 3, height_ratios=[1, 0.65], hspace=0.45, wspace=0.3)
        else:
            fig = plt.figure(figsize=(13, 5))
            gs  = fig.add_gridspec(1, 3, wspace=0.3)
        ax_yz   = fig.add_subplot(gs[0, 0])
        ax_xz   = fig.add_subplot(gs[0, 1])
        ax_xy   = fig.add_subplot(gs[0, 2])
        ax_hist = fig.add_subplot(gs[1, :]) if show_hist else None

        kw = dict(cmap=cmap, vmin=lo, vmax=hi, aspect='equal', origin='lower')

        im = ax_yz.imshow(data[x, :, :].T, **kw)
        ax_yz.axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        ax_yz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_yz.set_title(f'YZ  (x={x})')
        ax_yz.set_xlabel('Y');  ax_yz.set_ylabel('Z')

        ax_xz.imshow(data[:, y, :].T, **kw)
        ax_xz.axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        ax_xz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_xz.set_title(f'XZ  (y={y})')
        ax_xz.set_xlabel('X');  ax_xz.set_ylabel('Z')

        ax_xy.imshow(data[:, :, z].T, **kw)
        ax_xy.axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        ax_xy.axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        ax_xy.set_title(f'XY  (z={z})')
        ax_xy.set_xlabel('X');  ax_xy.set_ylabel('Y')

        ax_xy.legend(_crosshair_handles, _crosshair_labels,
                     fontsize=7, loc='lower right', framealpha=0.5, title='slice pos.')
        plt.colorbar(im, ax=[ax_yz, ax_xz, ax_xy], shrink=0.65, label=clabel)

        if show_hist:
            # Histogram: all K distributions overlaid, current k highlighted
            vol_interiors = [_interior(vol[k_]) for k_ in range(K_)]
            for k_ in range(K_):
                ax_hist.hist(
                    vol_interiors[k_], bins=bins, histtype='step',
                    color='steelblue' if k_ == k else 'gray',
                    lw=1.8            if k_ == k else 0.5,
                    alpha=1.0         if k_ == k else 0.25,
                    label=f'k={k_} (selected)' if k_ == k else '_nolegend_',
                )

            mu    = float(np.mean(vol_interiors[k]))
            sigma = float(np.std(vol_interiors[k]))
            ax_hist.axvline(mu,         color='steelblue', lw=1.5, ls='--', label=f'μ = {mu:.3g}')
            ax_hist.axvline(mu - sigma, color='steelblue', lw=1.0, ls=':', label=f'σ = {sigma:.3g}')
            ax_hist.axvline(mu + sigma, color='steelblue', lw=1.0, ls=':')
            ax_hist.axvline(_abs_lo, color='orange', lw=1.2, ls='--', alpha=0.8, label='cmap lo/hi')
            ax_hist.axvline(_abs_hi, color='orange', lw=1.2, ls='--', alpha=0.8)
            ax_hist.set_xlabel('Voxel intensity')
            ax_hist.set_ylabel('Count')
            ax_hist.set_title(
                f'Distributions — all K overlaid  (k={k},  μ={mu:.3g},  σ={sigma:.3g})'
            )
            ax_hist.legend(fontsize=8)

        plt.suptitle(
            f'{dataset}  |  k={k}  y_dir={np.round(y_directions[k], 2)}  —  {DATASET}/{DC_TYPE}',
            fontsize=11,
        )
        plt.show()

    interact(
        _view,
        dataset=widgets.Dropdown(
            options=keys, value=keys[0], description='Volume:',
            style={'description_width': 'initial'},
        ),
        show_difference=widgets.Checkbox(value=False, description=f'Diff from {ref_key}'),
        show_hist=widgets.Checkbox(value=show_histogram, description='Show histogram'),
        k=widgets.IntSlider(min=0, max=K_-1, step=1, value=0,
                            description='k (dir)', continuous_update=False),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


make_directional_viewer(volumes, y_directions)

Projecting SH → directional intensities...
  mumott_sh: (30, 141, 111, 141)
  mumott_gk: (30, 141, 111, 141)
  NAF: (30, 141, 111, 141)


interactive(children=(Dropdown(description='Volume:', options=('mumott_sh', 'mumott_gk', 'NAF'), style=Descrip…

---
## Optional: inject extra volumes

Extend `volumes` with any array not in the standard cache before calling either viewer.

In [ ]:
# Example — uncomment and adapt:
# volumes["GT"] = np.load("/myhome/data/smartt/shared/results/zenodo_benchmark/mumott_gk_zenodo_combined_1d296373.npy")
# volumes["GT"] = np.load("/myhome/data/smartt/shared/results/b411_benchmark/mumott_gk_b411_combined_c5dbd493.npy")
# volumes["NAF 100 remount"] = np.load("/myhome/data/smartt/shared/results/b411_benchmark/naf_b411_remount_d7bd20ed.npy")
volumes["GT"] = np.load(cache_dir / "ground_truth_bf0d2795.npy")
# volumes["NAF_old"] = np.load(cache_dir / "naf_synthetic-b411_main_9e3dd49c.npy")



# make_volume_viewer(volumes)
make_directional_viewer(volumes, y_directions)

---
## Viewer C — Principal Orientation

Extracts the principal eigenvector of the ODF at every voxel via
`SphericalHarmonics.get_output()` and maps its z-component to a 2-D heatmap.

- z-component ≈ 0  → fibre lies in the XY plane (in-plane)  
- z-component ≈ ±1 → fibre points along Z (out-of-plane)

Only SH-coefficient volumes (C = 45 for ell\_max=8) support this view.  
Any volume in `volumes` with the wrong C is skipped automatically.  
GK volumes appear here only if they were stored as SH-projected coefficients in the cache.

In [ ]:
from mumott.methods.basis_sets import SphericalHarmonics as _SHBasis

def make_orientation_viewer(volumes, ell_max=ELL_MAX):
    """Interactive principal-orientation viewer for SH-coefficient volumes."""
    expected_c = sum(2 * ell + 1 for ell in range(0, ell_max + 1, 2))
    sh_vols = {k: v for k, v in volumes.items() if v.shape[-1] == expected_c}

    if not sh_vols:
        print(f"No volumes with C={expected_c} (ell_max={ell_max}). Nothing to show.")
        return

    _basis = _SHBasis(ell_max=ell_max)
    dirs = {}
    for label, coeffs in sh_vols.items():
        out = _basis.get_output(coeffs)
        dirs[label] = torch.FloatTensor(out.eigenvector_1)  # (X, Y, Z, 3)
        print(f"  {label}: eigenvector_1 {dirs[label].shape}")

    keys = list(dirs.keys())
    X_, Y_, Z_, _ = dirs[keys[0]].shape
    z_hat = torch.tensor([0., 0., 1.])
    n = len(keys)

    def _view(x_slice):
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), squeeze=False)
        for col, label in enumerate(keys):
            z_comp = (dirs[label][x_slice] @ z_hat).numpy()  # (Y, Z)
            im = axes[0, col].imshow(z_comp.T, cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
            axes[0, col].set_title(label)
            axes[0, col].set_xlabel('Y')
            axes[0, col].set_ylabel('Z')
            plt.colorbar(im, ax=axes[0, col], fraction=0.046, label='z-component')

        plt.suptitle(
            f'Principal SH direction · ẑ  —  x={x_slice}  ({DATASET}/{DC_TYPE})\n'
            '≈ 0 → in-plane (XY)  |  ≈ ±1 → out-of-plane (Z)',
            fontsize=12,
        )
        plt.tight_layout()
        plt.show()

    interact(
        _view,
        x_slice=widgets.IntSlider(
            min=0, max=X_ - 1, step=1, value=X_ // 2,
            description='x slice', continuous_update=False,
        ),
    )


make_orientation_viewer(volumes)